<a href="https://colab.research.google.com/github/fleurclochette/Scientific-Project-SS26/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import subprocess
import numpy as np
import pandas as pd
import yfinance as yf
from google.colab import userdata

# Securely fetching credentials from Colab Secrets
GH_TOKEN = userdata.get('GITHUB_TOKEN')
GH_USER = userdata.get('GITHUB_USER')
GH_EMAIL = "tushar.latiyan@st.ovgu.de"
REPO_URL = f"https://{GH_TOKEN}@github.com/fleurclochette/Scientific-Project-SS26.git"

# Defining the exact 23-asset universe
tickers = [
    '^GSPC', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY',
    '^AXJO', '^DJI', '^FCHI', '^FTSE', '^GDAXI', '^GSPTSE', '^HSI',
    '^IXIC', '^KS11', '^N225', '^RUT', '^STOXX50E', '^TWII'
]

print("Pulling market data from Jan 2000 through July 2026...")

# Downloading data (auto_adjust=False prevents the KeyError)
df_raw = yf.download(tickers, start='2000-01-01', end='2026-08-01', auto_adjust=False)['Adj Close']

# Cleaning and calculating
# Forward fill to sync global holidays, but ONLY drop rows where ALL assets are missing (weekends)
prices = df_raw.ffill().dropna(how='all')

# Calculating daily log returns
log_returns = np.log(prices / prices.shift(1))

# Dropping only the very first row since shift(1) guarantees it is NaN for every asset on Day 1
log_returns = log_returns.iloc[1:]

# CROSS-CHECK: Verify the time frame explicitly before saving
start_date = prices.index.min().date()
end_date = prices.index.max().date()
print(f"\n--- DATA VALIDATION ---")
print(f"First Trading Day: {start_date}")
print(f"Last Trading Day:  {end_date}")

if str(start_date)[:4] == '2000':
    print("Start date check passed: Data begins in 2000.")
else:
    print(f"WARNING: Data begins in {start_date}.")

if str(end_date) == '2026-07-31':
    print("End date check passed: Data ends exactly on July 31, 2026.")
else:
    print(f"WARNING: End date is {end_date}, not July 31, 2026.")

print(f"Total Rows (Prices): {len(prices)}")
print(f"Total Rows (Returns): {len(log_returns)}")
print(f"-----------------------\n")

# Saving BOTH files locally
os.makedirs('data/raw', exist_ok=True)
prices_path = 'data/raw/master_23_assets_prices_2000_2026.csv'
returns_path = 'data/raw/master_23_assets_returns_2000_2026.csv'

prices.to_csv(prices_path)
log_returns.to_csv(returns_path)
print("Saved both raw prices and log returns locally.")

# Pushing both files securely to the main branch
print("Pushing both files securely to GitHub...")
cmds = [
    "rm -rf temp_repo",
    f"git clone {REPO_URL} temp_repo",
    "mkdir -p temp_repo/data/raw",
    f"cp {prices_path} temp_repo/data/raw/",
    f"cp {returns_path} temp_repo/data/raw/",
    f'cd temp_repo && git config user.email "{GH_EMAIL}" && git config user.name "{GH_USER}"',
    'cd temp_repo && git add data/raw/master_23_assets_prices_2000_2026.csv data/raw/master_23_assets_returns_2000_2026.csv',
    'cd temp_repo && git commit -m "fix: corrected dropna to preserve 2000-2007 data"',
    'cd temp_repo && git push origin main',
    "rm -rf temp_repo"
]

for cmd in cmds:
    subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Git push complete. The full 2000-2026 timeline is locked into the repository.")

[****                   9%                       ]  2 of 23 completed

Pulling market data from Jan 2000 through July 2026...


[*********************100%***********************]  23 of 23 completed



--- DATA VALIDATION ---
First Trading Day: 2000-01-03
Last Trading Day:  2026-07-31
Start date check passed: Data begins in 2000.
End date check passed: Data ends exactly on July 31, 2026.
Total Rows (Prices): 6916
Total Rows (Returns): 6915
-----------------------

Saved both raw prices and log returns locally.
Pushing both files securely to GitHub...
Git push complete. The full 2000-2026 timeline is locked into the repository.
